# Scholar Applications - Exploratory Data Analysis

**Data:** Source: internal Smartsheet data, 2025–26 application cycle

> **Data note:** The raw applicant data is proprietary and not included in this repository.
> See `data/README.md` for the full schema. Code is shown with outputs cleared.

---

## Research Context

Structure: partner universities nominate students from their applicant pool. This notebook explores the applicant dataset to:

1. Assess data quality - completeness, duplicates, invalid values
2. Describe the applicant pool - GPA, demographics, academic background, military affiliation
3. Surface outliers and distribution anomalies
4. Quantify correlations between key features
5. Prepare to model "What characteristics influence selection into the program?"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
#%pip install openpyxl
import openpyxl

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

RAW_PATH = '../data/raw/VSA_Dataset_20260316.xlsx'
SHEET    = 'Sheet1'

## 1. Data Loading

In [ ]:
raw = pd.read_excel(RAW_PATH, sheet_name=SHEET)
df  = raw.copy()

df["Created"] = pd.to_datetime(df["Created"])

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Date range: {df["Created"].min().date()} – {df["Created"].max().date()}')
df.head(10)

## 2. Variable Coverage & Missing Data

In [ ]:
coverage = (
    df.notna().mean().mul(100).round(1)
    .rename('pct_non_null')
    .to_frame()
    .assign(dtype=df.dtypes)
    .sort_values('pct_non_null')
    .reset_index()
    .rename(columns={'index': 'column'})
)
display(coverage)

## 3. Data Cleaning

Five cleaning operations are applied before exploratory analysis:
1. **Sparse row removal** - drop rows with ≥80% null columns (empty/test submissions)
2. **Deduplication** - some students submitted multiple records (ex: updated institution or degree type); keep most recent
3. **GPA normalization** - text values (`'No GPA yet'`, `'N/A'`, `'None'`) and zeros coerced to `NaN`
4. **Institution column** - consolidate `InstitutionName` (dropdown) and `Other_Institution` (free text) into a single `Institution` field
5. **Discipline column** - consolidate (`Discipline` and `Other Major` into a single Major field)

In [ ]:
# 1. Sparse row removal 
# Rows with ≥80% null columns are empty or test submissions with no usable data
sparse_mask = df.isna().mean(axis=1) >= 0.80
sparse_rows = df[sparse_mask]

print(f'Sparse rows (≥80% null): {sparse_mask.sum()}')
if len(sparse_rows):
    display(sparse_rows)

df = df[~sparse_mask].copy()
print(f'Records after sparse row removal: {len(df):,}')

In [ ]:
# 2. Deduplication
df['StudentID'] = df['StudentID'].astype(str).str.upper().str.strip()

n_before = len(df)
partial_dupes = df[df.duplicated(subset=['StudentID'], keep=False)]
print(f'Partial duplicates: {len(partial_dupes)} rows across '
      f'{partial_dupes["StudentID"].nunique()} students')

# Inspect what differs between duplicate submissions
dupe_diffs = []
for sid, grp in partial_dupes.groupby('StudentID'):
    differing = [c for c in grp.columns
                 if c not in ('StudentID',) and grp[c].nunique() > 1]
    if differing:
        dupe_diffs.append({'StudentID': sid, 'differing_columns': differing})

if dupe_diffs:
    display(pd.DataFrame(dupe_diffs))

# Keep most recent submission
df = df.sort_values('Created').drop_duplicates(subset=['StudentID'], keep='last').reset_index(drop=True)
print(f'Records after deduplication: {len(df):,} (removed {n_before - len(df)})')

In [ ]:
# 3. GPA normalization
# Known invalid text values found in raw data: 'No GPA yet', 'N/A', 'None' 
gpa_as_str = df['GPA'].astype(str).str.strip()
invalid_text_mask = gpa_as_str.isin(['no gpa yet', 'N/A', 'None', 'nan'])
invalid_text_counts = gpa_as_str[invalid_text_mask].value_counts()
if len(invalid_text_counts):
    print('Invalid text GPA values:')
    display(invalid_text_counts.rename('count').to_frame())


# catches all non-numeric strings
df['GPA'] = pd.to_numeric(df['GPA'], errors='coerce')

zero_gpa = (df['GPA'] == 0).sum()
df.loc[df['GPA'] == 0, 'GPA'] = np.nan

print(f'Zero GPAs to NaN: {zero_gpa}')
print(f'Total NaN GPA after cleaning: {df["GPA"].isna().sum()}')
print(f'GPA below 3.2 eligibility threshold: {(df["GPA"] < 3.2).sum()} '
      f'({(df["GPA"] < 3.2).mean():.1%})')

In [ ]:
# 4. Institution column 
# Consolidating free text 'Other' entry field for Institutions not listed in dropdown
# Note: full normalization (acronym expansion, spelling corrections, casing) is handled in the cleaning notebook

OTHERS = {
    'Other - My University is Not Listed',
    'Other - My Institution is Not Listed',
}

df['Institution'] = (
    df['InstitutionName']
    .replace(OTHERS, np.nan)
    .fillna(df['Other_Institution'])
)

n_sentinel       = df['InstitutionName'].isin(OTHERS).sum()
n_from_dropdown  = df['InstitutionName'].notna().sum() - n_sentinel
n_from_freetext  = df['Other_Institution'].notna().sum()
n_null_after     = df['Institution'].isna().sum()

print(f'From dropdown  (InstitutionName): {n_from_dropdown:,}')
print(f'From free text (Other Institution): {n_from_freetext:,}')
print(f'Null after merge: {n_null_after}')


In [ ]:
# 5. Discipline 
# Consolidating 'discipline' and 'other major' fields
df['Discipline'] = df['Discipline'].str.strip()

OTHER_DISCIPLINE = {'Other'}

# Count BEFORE replacing
n_other = df['Discipline'].isin(OTHER_DISCIPLINE).sum()

df['Discipline'] = (
    df['Discipline']
    .replace(OTHER_DISCIPLINE, np.nan)
    .fillna(df['OtherMajor'])
)

n_from_dropdown = df['Discipline'].notna().sum() - df['OtherMajor'].notna().sum()
n_from_freetext = df['OtherMajor'].notna().sum()
n_null_after    = df['Discipline'].isna().sum()

print(f'From dropdown (Discipline): {n_from_dropdown:,}')
print(f'From free text (OtherMajor): {n_from_freetext:,}')
print(f'Null after merge: {n_null_after}')

## 4. Distributions

### 4.1 Application Timing

In [ ]:
df['YearMonth'] = df['Created'].dt.to_period('M')
monthly = df.groupby('YearMonth').size().rename('count').reset_index()

fig, ax = plt.subplots(figsize=(9, 4))
sns.barplot(data=monthly, x='YearMonth', y='count', ax=ax, color='steelblue')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
ax.set_title('Applications per Month - 2025–26 FY')
ax.set_xlabel('')
ax.set_ylabel('Applications')
plt.tight_layout()
plt.show()

### 4.2 GPA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Overall distribution
sns.histplot(df['GPA'].dropna(), bins=25, kde=True, ax=axes[0])
axes[0].axvline(3.2, color='red', linestyle='--', linewidth=1.2, label='Eligibility threshold (3.2)')
axes[0].axvline(df['GPA'].mean(), color='navy', linestyle='--', linewidth=1.2,
                label=f'Mean ({df["GPA"].mean():.2f})')
axes[0].set_title('GPA Distribution — All Applicants')
axes[0].set_xlabel('GPA')
axes[0].legend(fontsize=8)

# By sex
sex_order = ['Male', 'Female']
sex_data = df[df['Sex'].isin(sex_order)]
sns.boxplot(data=sex_data, x='Sex', y='GPA', order=sex_order, ax=axes[1])
axes[1].set_title('GPA by Sex')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

display(df['GPA'].describe().rename('GPA').to_frame().T)

In [ ]:
# GPA by top institutions and disciplines
def boxplot_by_top(df, group_col, top_n=10, title_suffix=''):
    sub = df[(df['GPA'].notna()) & (df[group_col].notna())]
    top = sub[group_col].value_counts().head(top_n).index
    sub = sub[sub[group_col].isin(top)]
    order = sub.groupby(group_col)['GPA'].median().sort_values(ascending=False).index

    fig, ax = plt.subplots(figsize=(11, 4))
    sns.boxplot(data=sub, x=group_col, y='GPA', order=order, ax=ax, color = 'steelblue')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=40, ha='right', fontsize=8)
    ax.set_title(f'GPA Distribution - Top {top_n} {title_suffix}')
    ax.set_xlabel('')
    plt.tight_layout()
    plt.show()

boxplot_by_top(df, 'Discipline', top_n=8, title_suffix='Disciplines')
boxplot_by_top(df, 'Institution', top_n=10, title_suffix='Institutions')

### 4.3 Demographics

In [ ]:
def freq_bar(df, col, title, ax, top_n=None, dropna=True):
    counts = df[col].value_counts(dropna=dropna)
    if top_n:
        counts = counts.head(top_n)
    pct = counts / counts.sum() * 100
    bars = ax.barh(counts.index[::-1], pct.values[::-1])
    for bar, p in zip(bars, pct.values[::-1]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{p:.1f}%', va='center', fontsize=8)
    ax.set_xlabel('% of applicants')
    ax.set_title(title)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter())

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
freq_bar(df, 'Sex', 'Sex', axes[0])
freq_bar(df, 'Race', 'Race/Ethnicity', axes[1])
freq_bar(df, 'CountryOfCitizenship', 'Citizenship', axes[2])
plt.tight_layout()
plt.show()

### 4.4 Academic Profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
freq_bar(df, 'DegreeType', 'Degree Type', axes[0])
freq_bar(df, 'StudentClass', 'Student Year', axes[1])
freq_bar(df, 'Discipline', 'Discipline (top 8)', axes[2], top_n=8)
plt.tight_layout()
plt.show()

print(f'\nTop 5 institutions by application volume:')
display(df['Institution'].value_counts().head(5).rename('count').to_frame())

### 4.5 Military Affiliation & DoD Program Variables

In [ ]:
# Derive military affiliation flag
cadet_flag = df.get('CadetCorpTF', pd.Series(dtype=str)).eq('Yes')
rotc_flag  = df.get('ROTCBranch', pd.Series(dtype=str)).notna() & \
             df.get('ROTCBranch', pd.Series(dtype=str)).ne('') & \
             df.get('ROTCBranch', pd.Series(dtype=str)).ne('nan')
df['mil_affil'] = (cadet_flag | rotc_flag).astype(int)

# DoD scholarship composite
df['dod_scholar'] = (
    df.get('SFSTF', pd.Series(dtype=str)).eq('Yes') |
    df.get('SMARTTF', pd.Series(dtype=str)).eq('Yes') |
    df.get('CSATF', pd.Series(dtype=str)).eq('Yes')
).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
freq_bar(df, 'ROTCBranch', 'ROTC Branch', axes[0])
freq_bar(df, 'DoD_Job_Interest', 'DoD_Job_Interest', axes[1])
freq_bar(df, 'CadetCorpTF', 'Corps of Cadets', axes[2])
plt.tight_layout()
plt.show()

print(f'ROTC/cadet affiliated: {df["mil_affil"].sum()} ({df["mil_affil"].mean():.1%})')
print(f'DoD scholarship (SFS/SMART/CSA): {df["dod_scholar"].sum()} ({df["dod_scholar"].mean():.1%})')

In [ ]:
# Institution concentration check 
inst_counts = df['Institution'].value_counts()
top_inst_pct = inst_counts.head(10).sum() / len(df)
print(f'\nTop 10 institutions share: {top_inst_pct:.1%} of all applications')

In [ ]:
# Visualize: application count per institution (sorted)
fig, ax = plt.subplots(figsize=(10, 4))
inst_counts_sorted = inst_counts.sort_values(ascending=False).reset_index()
inst_counts_sorted.columns = ['institution', 'count']
ax.bar(range(len(inst_counts_sorted)), inst_counts_sorted['count'], width=1.0, edgecolor='none')
ax.set_xlabel('Institution (ranked by application volume)')
ax.set_ylabel('Applications')
ax.set_title('Application Volume per Institution')
plt.tight_layout()
plt.show()

## 6. Correlation Analysis

In [ ]:
# Build a numeric analysis frame for correlation
corr_df = pd.DataFrame({
    'GPA':             df['GPA'],
    'female':          (df['Sex'] == 'Female').astype(float),
    'pell':            (df['PellGrantTF'] == 'Yes').astype(float),
    'veteran':         (df['VeteranTF'] == 'Yes').astype(float),
    'mil_affil':       df['mil_affil'].astype(float),
    'dod_scholar':     df['dod_scholar'].astype(float),
    'dod_interest_hi': (df['DoD_Job_Interest'] == 'Yes').astype(float),
    'grad_student':    (df['DegreeType'].str.contains('Master|PhD|Doctor', na=False)).astype(float),
    'us_citizen':      (df['CountryOfCitizenship'] == 'US Citizen').astype(float),
    'cs_cyber':        (df['Discipline'].isin(['Computer Science', 'Cybersecurity'])).astype(float),
})

corr_matrix = corr_df.corr(method='pearson')

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-0.5, vmax=0.5,
    linewidths=0.5, ax=ax
)
ax.set_title('Pairwise Correlations — Key Applicant Features\n(lower triangle only)')
plt.tight_layout()
plt.show()

In [ ]:
# Highlight the stronger pairwise correlations (absolute r > 0.15)
corr_long = (
    corr_matrix.where(~mask)
    .stack()
    .reset_index()
    .rename(columns={0: 'r', 'level_0': 'var1', 'level_1': 'var2'})
    .assign(abs_r=lambda x: x['r'].abs())
    .query('abs_r > 0.15')
    .sort_values('abs_r', ascending=False)
)
print('Notable correlations (|r| > 0.10):')
display(corr_long[['var1', 'var2', 'r']].reset_index(drop=True))

## 7. Key Findings 

**Note: Some findings and specifics removed for privacy.**

* High-achieving pool
* Male dominated
* Over half of applicants study Computer Science or Cybersecurity
